# Evaluating a knowledge edit — efficacy vs. specificity (Qwen3-0.6B, T4)

One prompt is easy to change. The hard part of knowledge editing is changing it
**without breaking the neighbours**. This notebook builds a tagged evaluation
battery, edits the `France -> Paris` feature constellation on the live model, and
measures the trade-off:

| axis | editing-literature name | tag here |
|---|---|---|
| did the target change? | **efficacy** | `target` |
| did related facts survive? | **specificity** | `neighbour` |
| did unrelated facts survive? | **locality** | `control` |

Then: sweep the edit strength to trace the **Pareto frontier**, and compare
`suppress` vs `ablate` vs `steer`.

Runtime: **T4 GPU**.


In [ ]:
!pip install -q 'transformers>=4.51' accelerate safetensors matplotlib
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

In [ ]:
import torch, numpy as np, marv
from transformers import AutoModelForCausalLM, AutoTokenizer
device = 'cuda' if torch.cuda.is_available() else 'cpu'

NAME = 'Qwen/Qwen3-0.6B'
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=torch.float16).to(device).eval()

vindex = marv.extract(model, model_name=NAME)
marv.build_down_meta(vindex)
print('bands:', vindex.layer_bands)

## The evaluation battery

`target` includes **rephrasings** — an edit that only works on the exact training
prompt but fails on "The French capital is ___" has poor *generalization*, a known
failure mode. `neighbour` probes share the constellation. `control` is unrelated.


In [ ]:
P = marv.Probe
battery = [
    # --- target: the fact + rephrasings (generalization) ---
    P('The capital of France is', 'Paris', ('target',)),
    P('Paris is the capital of', 'France', ('target',)),
    P('The French capital city is called', 'Paris', ('target',)),
    P('Q: What is the capital of France? A:', 'Paris', ('target',)),

    # --- neighbours: share the constellation (collateral risk) ---
    P('The capital of Italy is', 'Rome', ('neighbour', 'capital')),
    P('The capital of Spain is', 'Madrid', ('neighbour', 'capital')),
    P('The capital of Germany is', 'Berlin', ('neighbour', 'capital')),
    P('The capital of Portugal is', 'Lisbon', ('neighbour', 'capital')),
    P('The official language of France is', 'French', ('neighbour', 'france')),
    P('The currency of France is the', 'euro', ('neighbour', 'france')),
    P('The Eiffel Tower is located in', 'Paris', ('neighbour', 'paris')),
    P('The Louvre museum is in', 'Paris', ('neighbour', 'paris')),

    # --- controls: unrelated, must not move ---
    P('The capital of Japan is', 'Tokyo', ('control', 'capital')),
    P('The capital of Brazil is', 'Brasilia', ('control', 'capital')),
    P('Water is made of hydrogen and', 'oxygen', ('control', 'science')),
    P('The chemical symbol for gold is', 'Au', ('control', 'science')),
    P('The opposite of hot is', 'cold', ('control', 'lexical')),
    P('The plural of mouse is', 'mice', ('control', 'lexical')),
    P('2 + 2 =', '4', ('control', 'math')),
    P('The first president of the United States was', 'George', ('control', 'history')),
]
print(len(battery), 'probes')

## Baseline — does the model actually know these?

An edit eval is only meaningful on facts the model gets right unedited.


In [ ]:
base = marv.run_battery(model, tok, battery, device=device)
for r in base.rows:
    ok = 'ok' if r.target_rank == 1 else f'r{r.target_rank}'
    print(f'  [{ok:>4}] p={r.target_prob:.3f}  {r.prompt!r} -> {r.top1!r} (want {r.target!r}) {r.tags}')

## Locate the `France` constellation


In [ ]:
con = marv.constellation(vindex, tok, 'France', per_layer=4)
for r in con[:14]:
    print(f'  L{r.layer:>2} f{r.feature:<5} sim={r.sim:.2f}  -> {r.tokens[:3]}')
feats = [(r.layer, r.feature) for r in con]

## One edit: suppress the top 5, read the table


In [ ]:
rep = marv.study_edit(model, tok, marv.suppress(model, feats[:5]), battery, device=device)
rep.show()
print()
for tag, m in sorted(rep.metrics().items()):
    print(f'  {tag:<12} n={int(m["n"]):>2}  moved={m["moved"]:.2f}  mean dprob={m["mean_dprob"]:+.3f}')

## The frontier — sweep edit strength

More features suppressed = bigger hit to the target, but more collateral. Is there
a window where the target drops and the neighbours don't?


In [ ]:
sizes = [0, 1, 2, 3, 4, 6, 8, 10, 14]
sweep = marv.suppression_frontier(model, tok, feats, battery, sizes=sizes, device=device)

rows = []
for n, d in sweep:
    m = d.metrics()
    g = lambda t, k: m.get(t, {}).get(k, 0.0)
    rows.append((n, g('target','mean_dprob'), g('neighbour','mean_dprob'), g('control','mean_dprob'),
                 g('neighbour','moved'), g('control','moved')))
print(f'{"n":>3} {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10} {"neigh mv":>9} {"ctrl mv":>8}')
for r in rows:
    print(f'{r[0]:>3} {r[1]:>+10.3f} {r[2]:>+10.3f} {r[3]:>+10.3f} {r[4]:>9.2f} {r[5]:>8.2f}')

In [ ]:
import matplotlib.pyplot as plt
ns = [r[0] for r in rows]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ns, [-r[1] for r in rows], 'o-', label='target (efficacy)')
ax[0].plot(ns, [-r[2] for r in rows], 's-', label='neighbour (collateral)')
ax[0].plot(ns, [-r[3] for r in rows], '^-', label='control (collateral)')
ax[0].set_xlabel('features suppressed'); ax[0].set_ylabel('mean target-prob DROP')
ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot([-r[2] for r in rows], [-r[1] for r in rows], 'o-')
for r in rows: ax[1].annotate(str(r[0]), (-r[2], -r[1]), fontsize=8, xytext=(3,3), textcoords='offset points')
ax[1].set_xlabel('neighbour prob drop (collateral)'); ax[1].set_ylabel('target prob drop (efficacy)')
ax[1].set_title('Pareto frontier'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## `suppress` vs `ablate` vs `steer`

- **suppress** — zero the feature activations (forward hook, reversible)
- **ablate** — zero `down_proj[:, f]` in the weights (permanent)
- **steer** — push the residual away from the `Paris` direction at one layer


In [ ]:
edit_feats = feats[:5]
sup = marv.study_edit(model, tok, marv.suppress(model, edit_feats), battery, device=device).metrics()

b0 = marv.run_battery(model, tok, battery, device=device)
saved = marv.ablate(model, edit_feats)
abl = marv.diff_battery(b0, marv.run_battery(model, tok, battery, device=device)).metrics()
marv.restore(model, saved)

paris_id = tok.encode(' Paris', add_special_tokens=False)[0]
direction = vindex.embed[paris_id].astype(np.float32)
kb = vindex.band('knowledge'); mid = kb[len(kb)//2]
ste = marv.study_edit(model, tok, marv.steer(model, mid, direction, alpha=-8.0), battery, device=device).metrics()

print(f'{"":<10} {"target dp":>10} {"neigh dp":>10} {"ctrl dp":>10}')
for name, m in [('suppress', sup), ('ablate', abl), ('steer', ste)]:
    g = lambda t: m.get(t, {}).get('mean_dprob', 0.0)
    print(f'{name:<10} {g("target"):>+10.3f} {g("neighbour"):>+10.3f} {g("control"):>+10.3f}')

## Localization — does it matter *which* layer you edit?

Suppress only the `France` features found at each single knowledge layer and
measure the target hit. (Per Hase et al., where a fact *is* stored is not
necessarily where an edit works best.)


In [ ]:
layer_effect = []
for L in vindex.band('knowledge'):
    rows_L = marv.describe_entity(vindex, tok, 'France', layers=[L], k_features=6)
    fL = [(r.layer, r.feature) for r in rows_L]
    d = marv.study_edit(model, tok, marv.suppress(model, fL), battery, device=device).metrics()
    layer_effect.append((L, d['target']['mean_dprob'], d.get('neighbour', {}).get('mean_dprob', 0.0)))

Ls = [x[0] for x in layer_effect]
plt.figure(figsize=(10, 3.5))
plt.bar([l-0.2 for l in Ls], [-x[1] for x in layer_effect], width=0.4, label='target drop')
plt.bar([l+0.2 for l in Ls], [-x[2] for x in layer_effect], width=0.4, label='neighbour drop')
plt.xlabel('layer edited (in isolation)'); plt.ylabel('mean prob drop'); plt.legend(); plt.grid(alpha=.3)
plt.show()

## Reading the result

- A **knee** in the frontier (target drops, neighbours still flat) is your operating
  point. A straight line through the origin means the fact and its neighbours share
  the same features — no clean edit, expected for a well-connected entity.
- `ablate` should roughly match `suppress`. `steer` is blunter — usually more
  collateral for the same efficacy.
- Paper-grade eval would add: multi-hop consequences (MQuAKE-style), a bigger
  control set, several target facts averaged, and a fluency check on free generation.

### Next
- Swap `France` for a rare entity (`Liechtenstein`, a fictional place) — collateral
  usually drops because the constellation is less shared.
- Run this battery on `(Qwen3-0.6B-Base, Qwen3-0.6B)` and correlate the facts
  post-training changed with `marv.diff`'s most-moved features.
